In [1]:
from proj_utils import *

import os
import pickle
from os.path import join
from copy import deepcopy
from collections import defaultdict, Counter
from itertools import combinations
from datetime import datetime
from dateutil.relativedelta import relativedelta
import ipywidgets as widgets
from IPython.display import display

import numpy as np
import pandas as pd
from tqdm import tqdm

import sklearn as sk
from scipy.optimize import linear_sum_assignment
from scipy.optimize import curve_fit
from sklearn.metrics.pairwise import cosine_similarity

import torch
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import ipywidgets as widgets
from ipywidgets import interact, interact_manual

import bertopic
from sentence_transformers import SentenceTransformer

from proj_utils import *

/home/seungwoong.ha/anaconda3/envs/collmind/lib/python3.11/site-packages/umap/distances.py:1063: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/home/seungwoong.ha/anaconda3/envs/collmind/lib/python3.11/site-packages/umap/distances.py:1071: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/home/seungwoong.ha/anaconda3/envs/collmind/lib/pyth

### Caution : before run this, run [transform] in 'fit-final-models.py' and time-analysis.py.

In [7]:
collection_name = 'Thehill'
model_name = MODEL_NAMES[collection_name]
threshold = 10

start_date, end_date = DATE_RANGES[collection_name]
num_topics = NUM_TOPICS[collection_name]
topic_model = (BERTopic.load(join('model', collection_name.lower(), model_name), embedding_model="all-MiniLM-L6-v2"))

In [3]:
# open pickle files
with open(join('article', collection_name.lower(), MODEL_NAMES[collection_name], 'article_dict.pkl'), 'rb') as f:
    article_dict = pickle.load(f)

In [ ]:
# load articles_by_day from feather files and remove articles below threshold

articles_by_day = {}

file_names = os.listdir(join('data', 'collmind', 'article', collection_name.lower(), model_name, 'articles_by_day'))
for file_name in file_names:
    day = file_name.split('.')[0]
    print(day)
    articles = pd.read_feather(join('article', collection_name.lower(), model_name, 'articles_by_day', file_name))
    articles_by_day[day] = articles[articles['comment_id'].apply(len) > threshold]

In [ ]:
date_cutoff = 7

for article_day in articles_by_day.keys():
    print(article_day)
    articles = articles_by_day[article_day]
    comment_createdAt_list = []
    comment_id_list = []
    comment_topics_list = []
    comments_embeddings_list = []
    for article in articles.itertuples():
        filtered_index = np.where(np.array([(comment_createdAt - article.createdAt).days for comment_createdAt in article.comment_createdAt]) < date_cutoff)[0]
        comment_createdAt_list.append([article.comment_createdAt[i] for i in filtered_index])
        comment_id_list.append([article.comment_id[i] for i in filtered_index])
        comment_topics_list.append([article.comment_topics[i] for i in filtered_index])
        comments_embeddings_list.append(np.array([article.comments_embeddings[i] for i in filtered_index]))

    articles_by_day[article_day] = articles_by_day[article_day].assign(comment_id=comment_id_list, comment_topics=comment_topics_list, comment_createdAt=comment_createdAt_list, comments_embeddings=comments_embeddings_list)

In [ ]:
# merge articles_by_day into articles_by_'month' (not month, but given time interval)

aggregate_days = 7

articles_by_month = dict()
keys = sorted(articles_by_day.keys(), key=lambda x: datetime.strptime(x, '%Y-%m-%d'))

start = datetime.strptime(keys[0], '%Y-%m-%d')
index = 0
while True:
    end = start + relativedelta(days=aggregate_days)
    print(start, end)
    if end > datetime.strptime(keys[-1], '%Y-%m-%d'):
        break
    tmp_list = []
    while True:
        if datetime.strptime(keys[index], '%Y-%m-%d') >= end:
            break
        else:
            tmp_list.append(articles_by_day[keys[index]])
            del articles_by_day[keys[index]]  # for the memory
            index += 1
    if len(tmp_list) > 0:
        articles_by_month[start.strftime('%Y-%m-%d')] = pd.concat(tmp_list)
    start = end

## 1. articles_by_day from article_dict

In [ ]:
# from pymongo collection, query articles with given id
def get_articles_from_ids(collection, ids, select_columns):
    query = {'_id': {'$in': ids}}
    projection = {k: 1 for k in select_columns}
    articles = collection.find(query, projection)
    return pd.DataFrame(articles)

# from pymongo collection, query articles with given date range
def get_articles_from_date(collection, start_date, end_date, select_columns):
    query = {'createdAt': {'$gte': start_date, '$lt': end_date}}
    projection = {k: 1 for k in select_columns}
    articles = collection.find(query, projection)
    return pd.DataFrame(articles)

# get comments with given article id
def get_comments_from_id(collection, id, select_columns):
    query = {'art_id': id}
    projection = {k: 1 for k in select_columns}
    comments = collection.find(query, projection)
    return pd.DataFrame(comments)

In [ ]:
def get_articles_by_day(collection_articles, collection_comments, start_date, end_date, select_columns):
    device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
    sentence_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
    articles_by_day = defaultdict(list)
    while start_date < end_date:
        next_day = start_date + relativedelta(days=1)
        articles = get_articles_from_date(collection_articles, start_date, next_day, select_columns)
        if len(articles) > 0:
            articles['title_embeddings'] = sentence_model.encode(articles['clean_title'].values.tolist(), show_progress_bar=True, convert_to_tensor=True).tolist()
        articles_by_day[start_date.strftime("%Y-%m-%d")] = articles
        start_date = next_day
    return articles_by_day

In [ ]:
from proj_utils import _init_mongo_collection

select_columns = ['_id', 'clean_title', 'createdAt']
mongo_client_articles, collection_articles = _init_mongo_collection('Articles', collection_name)
mongo_client_comments, collection_comments = _init_mongo_collection('Comments', collection_name)

### A. Query article title info from mongoDB

In [ ]:
articles_by_day = get_articles_by_day(collection_articles, collection_comments, start_date, end_date, select_columns)

# pop element from articles_by_day if there are no articles
for day in list(articles_by_day.keys()):
    if len(articles_by_day[day]) == 0:
        articles_by_day.pop(day)

### B. The guardians of time (relocating time-traveling articles into its rightful position)

In [ ]:
# load articles_by_day from pickle file
with open(join('article', collection_name.lower(), model_name, 'articles_by_day.pkl'), 'rb') as f:
    articles_by_day = pickle.load(f)

In [ ]:
time_traveler_df = pd.DataFrame(columns=articles_by_day[list(articles_by_day.keys())[0]].columns)
move_day_list = []

original_key_list = list(articles_by_day.keys())

for article_day in original_key_list:
    print(article_day)
    articles = articles_by_day[article_day]
    article_date = datetime.strptime(article_day, '%Y-%m-%d').strftime('%m%y')

    # remove time travelers

    for article_id, article_createdAt in articles[['_id', 'createdAt']].values:
        if article_id in article_dict.keys():
            article = article_dict[article_id]
            
            # sort article['createdAt'] and return sorted index as well
            sorted_index = np.argsort(article['createdAt'])
            ids = np.array(article['id'])[sorted_index]
            comment_createdAt_list = np.array(article['createdAt'])[sorted_index]
            article_dict[article_id]['article_createdAt'] = article_createdAt
            
            # check whether the where the comments_createdAt is earlier thatn the createdAt, check those indices.
            time_check = np.where(np.array([(comment_createdAt - article_createdAt).days for comment_createdAt in comment_createdAt_list]) < 0)[0]
            if len(time_check) > 0:
                time_traveler = articles[articles['_id'] == article_id]
                
                articles_by_day[article_day] = articles_by_day[article_day][articles_by_day[article_day]['_id'] != article_id] # pop time traveler from current day
                time_traveler_df = time_traveler_df.append(time_traveler)
                first_comment_date = comment_createdAt_list[0]
                move_day_list.append(first_comment_date)
                
                # change the createdAt of the article
                new_article_createdAt = first_comment_date - relativedelta(days=1)
                time_traveler['createdAt'] = new_article_createdAt
                article_dict[article_id]['article_createdAt'] = new_article_createdAt

                # append time traveler to new day
                new_article_day = first_comment_date.strftime('%Y-%m-%d')
                articles_by_day[new_article_day] = articles_by_day[new_article_day].append(time_traveler)
                
                # sort articles_by_day[new_article_day] by createdAt and reindex
                articles_by_day[new_article_day] = articles_by_day[new_article_day].sort_values(by='createdAt').reset_index(drop=True)
                
            article_dict[article_id]

time_traveler_df['move_day'] = move_day_list

In [ ]:
# load time_travler_df.feather
time_traveler_df = pd.read_feather(join('article', collection_name.lower(), model_name, 'time_traveler_df.feather'))
time_traveler_df

In [ ]:
# save time_traveler_df
time_traveler_df.reset_index().to_feather(join('article', collection_name.lower(), model_name, 'time_traveler_df.feather'))
# save article_dict again
with open(join('article', collection_name.lower(), model_name, 'article_dict.pkl'), 'wb') as f:
    pickle.dump(article_dict, f)
# save articles_by_day to pickle file
with open(join('article', collection_name.lower(), model_name, 'articles_by_day.pkl'), 'wb') as f:
    pickle.dump(articles_by_day, f)

### C. Add title topic classification / comments data into article_by_days

In [4]:
# load articles_by_day from pickle file
with open(join('article', collection_name.lower(), model_name, 'articles_by_day.pkl'), 'rb') as f:
    articles_by_day = pickle.load(f)

In [5]:
def load_embeddings(date):
    print(date)
    next_date = next_month(date)
    embedding_foler = '/data/comments/valentin/sentence-embeddings/'
    embedding_path = embedding_foler + f'{collection_name.lower()}/bert-emb-{date}-{next_date}.pt'
    embedding = torch.load(embedding_path, map_location=torch.device('cpu'))
    if embedding['embeddings'].device.type == 'cuda':
        print(date, 'cuda')
        embedding['embeddings']= embedding['embeddings'].to('cpu')
    return embedding

In [8]:
first=True
current_date = datetime.strptime(list(articles_by_day.keys())[0], '%Y-%m-%d').strftime('%m%y')
next_date = next_month(current_date)

original_key_list = list(articles_by_day.keys())

for article_day in original_key_list:
    
    print(article_day)
    articles = articles_by_day[article_day]
    article_date = datetime.strptime(article_day, '%Y-%m-%d').strftime('%m%y')
    
    # check whether the file named as articly_day.feather exist

    if os.path.isfile(join('article', collection_name.lower(), model_name, 'articles_by_day', article_day + '.parquet')):
        print('Already processed', article_day)
        articles_by_day.pop(article_day)
        continue
    
    # load embeddings

    if first:
        current_date = datetime.strptime(article_day, '%Y-%m-%d').strftime('%m%y')
        next_date = next_month(current_date)
        current_embeddings = load_embeddings(current_date)
        next_embeddings = load_embeddings(next_month(current_date)) 
        merged_embeddings = pd.DataFrame()
        merged_embeddings['_id'] = current_embeddings['_id'] + next_embeddings['_id']
        merged_embeddings['embeddings'] = torch.vstack((current_embeddings['embeddings'], next_embeddings['embeddings'])).numpy().tolist()
        
        first = False
        
    if current_date != article_date:  # next month happened)
    
        current_date = article_date
        next_date = next_month(current_date)
        current_embeddings = next_embeddings
        next_embeddings = load_embeddings(next_date)
        merged_embeddings = pd.DataFrame()
        merged_embeddings['_id'] = current_embeddings['_id'] + next_embeddings['_id']
        merged_embeddings['embeddings'] = torch.vstack((current_embeddings['embeddings'], next_embeddings['embeddings'])).numpy().tolist()

    # title topic

    class_nums = []
    class_probs = []

    for i in range(len(articles)):
        output = topic_model.find_topics(articles['clean_title'].values[i])
        class_nums.append(output[0])
        class_probs.append(output[1])
        
    articles['topic_num'] = class_nums
    articles['topic_prob'] = class_probs
    
    # comments data
    
    id_list = []
    topic_list = []
    createdAt_list = []
    embeddings_list = []
    original_comment_num_list = []
    
    for article_id, article_createdAt in articles[['_id', 'createdAt']].values:
        if article_id in article_dict.keys():
            article = article_dict[article_id]
            # sort article['createdAt'] and return sorted index as well
            sorted_index = np.argsort(article['createdAt'])
            ids = np.array(article['id'])[sorted_index]
            
            ## ~360 times faster than np.isin! (assuming that merged embeddings / ids are unique and sorted, which they are)
            id_indices = np.intersect1d(merged_embeddings['_id'], ids, assume_unique=True, return_indices=True)[1]
            embeddings_list.append(merged_embeddings.iloc[id_indices]['embeddings'].tolist())
            # from merged_embeddings, remove the embeddings that are already used (thehill, after 20190901)
            merged_embeddings = merged_embeddings.drop(id_indices)
            merged_embeddings.reset_index(drop=True, inplace=True)
            len_cutoff = len(embeddings_list[-1]) # maximum 2 months 

            id_list.append(np.array(article['id'])[sorted_index][:len_cutoff])
            topic_list.append(np.array(article['topics'])[sorted_index][:len_cutoff])
            createdAt_list.append(np.array(article['createdAt'])[sorted_index][:len_cutoff])
            original_comment_num_list.append(ids)
            assert len(embeddings_list[-1]) == len(id_list[-1]) == len(topic_list[-1]) == len(createdAt_list[-1])
            
        else:
            id_list.append([])
            topic_list.append([])
            createdAt_list.append([])
            embeddings_list.append([])
            
    articles['comment_id'] = id_list
    articles['comment_topics'] = topic_list
    articles['comment_createdAt'] = createdAt_list
    articles['comment_embeddings'] = embeddings_list
    
    articles.reset_index(drop=True).to_parquet(join('article', collection_name.lower(), model_name, 'articles_by_day', article_day + '.parquet'), compression='gzip')
    articles_by_day.pop(article_day)
    del id_list, topic_list, createdAt_list, embeddings_list, articles

2020-07-22
0720
0820
2020-07-23
Already processed 2020-07-23
2020-07-24
Already processed 2020-07-24
2020-07-25
Already processed 2020-07-25
2020-07-26
Already processed 2020-07-26
2020-07-27
Already processed 2020-07-27
2020-07-28
Already processed 2020-07-28
2020-07-29
Already processed 2020-07-29
2020-07-30
Already processed 2020-07-30
2020-07-31
Already processed 2020-07-31
2020-08-01
Already processed 2020-08-01
2020-08-02
Already processed 2020-08-02
2020-08-03
Already processed 2020-08-03
2020-08-04
Already processed 2020-08-04
2020-08-05
Already processed 2020-08-05
2020-08-06
Already processed 2020-08-06
2020-08-07
Already processed 2020-08-07
2020-08-08
Already processed 2020-08-08
2020-08-09
Already processed 2020-08-09
2020-08-10
Already processed 2020-08-10
2020-08-11
Already processed 2020-08-11
2020-08-12
Already processed 2020-08-12
2020-08-13
Already processed 2020-08-13
2020-08-14
Already processed 2020-08-14
2020-08-15
Already processed 2020-08-15
2020-08-16
Already 

## 2. Adding sentiments into articles_by_day

In [9]:
def load_sentiments(collection_name, date):
    print(date)
    next_date = next_month(date)
    sentiments_foler = f'/data/comments/valentin/max-sentiment-analysis/{collection_name.lower()}/sentiments_by_comment/'
    sentiments_path = sentiments_foler + f'batch-{date}.csv_createdAt'
    sentiments = pd.read_csv(sentiments_path)  # order : anger	anticipation	disgust	fear	joy	love	optimism	pessimism	sadness	surprise	trust
    return sentiments

In [10]:
original_key_list = os.listdir(join('article', collection_name.lower(), model_name, 'articles_by_day'))
original_key_list = [file_name.split('.')[0] for file_name in original_key_list]
original_key_list = sorted(original_key_list, key=lambda x: datetime.strptime(x, '%Y-%m-%d'))

first=True
current_date = datetime.strptime(original_key_list[0], '%Y-%m-%d').strftime('%m%y')
next_date = next_month(current_date)

for article_day in original_key_list:
    print(article_day) 
    
    with open(join('article', collection_name.lower(), model_name, 'articles_by_day', article_day + '.parquet'), 'rb') as f:
        articles = pd.read_parquet(f)
        if 'comment_sentiments' in articles.columns:
            print('Already processed', article_day)
            continue
    
    article_date = datetime.strptime(article_day, '%Y-%m-%d').strftime('%m%y')
    
    if first:
        current_date = datetime.strptime(article_day, '%Y-%m-%d').strftime('%m%y')
        next_date = next_month(current_date)
        current_sentiments = load_sentiments(collection_name, current_date)
        next_sentiments = load_sentiments(collection_name, next_month(current_date)) 
        merged_sentiments = pd.concat([current_sentiments, next_sentiments], ignore_index=True)
        merged_sentiments.set_index('id', inplace=True)
        first = False
        
    if current_date != article_date:  # next month happened)
        current_date = article_date
        next_date = next_month(current_date)
        current_sentiments = load_sentiments(collection_name, current_date)
        next_sentiments = load_sentiments(collection_name, next_month(current_date)) 
        merged_sentiments = pd.concat([current_sentiments, next_sentiments], ignore_index=True)
        merged_sentiments.set_index('id', inplace=True)

    sentiments_list = []
    
    for comment_id in articles['comment_id']:
        if len(comment_id):
            id_list = [int(c) for c in comment_id]
            sentiments_list.append(merged_sentiments.loc[id_list][['anger', 'anticipation', 'disgust', 'fear', 'joy', 'love', 'optimism', 'pessimism', 'sadness', 'surprise', 'trust']].values.tolist())
        else:
            sentiments_list.append([])

    articles['comment_sentiments'] = sentiments_list
    articles.to_parquet(join('article', collection_name.lower(), model_name, 'articles_by_day', article_day + '.parquet'), compression='gzip')

2012-07-19
Already processed 2012-07-19
2012-07-20
Already processed 2012-07-20
2012-07-21
Already processed 2012-07-21
2012-07-22
Already processed 2012-07-22
2012-07-23
Already processed 2012-07-23
2012-07-24
Already processed 2012-07-24
2012-07-25
Already processed 2012-07-25
2012-07-26
Already processed 2012-07-26
2012-07-27
Already processed 2012-07-27
2012-07-28
Already processed 2012-07-28
2012-07-29
Already processed 2012-07-29
2012-07-30
Already processed 2012-07-30
2012-07-31
Already processed 2012-07-31
2012-08-01
Already processed 2012-08-01
2012-08-02
Already processed 2012-08-02
2012-08-03
Already processed 2012-08-03
2012-08-04
Already processed 2012-08-04
2012-08-05
Already processed 2012-08-05
2012-08-06
Already processed 2012-08-06
2012-08-07
Already processed 2012-08-07
2012-08-08
Already processed 2012-08-08
2012-08-09
Already processed 2012-08-09
2012-08-10
Already processed 2012-08-10
2012-08-11
Already processed 2012-08-11
2012-08-12
Already processed 2012-08-12


## 3. Constructing sampling_dict for global / title models

### A. global sampling dict

In [ ]:
seed_dict = {'Atlantic': 4,
             'Breitbart': 3,
             'Gatewaypundit': 4,
             'Motherjones': 5,
             'Thehill': 2}

sampling_size = int(2e6/5)

for i in range(5):
    seed_global = i+1
    print(seed_global)
    
    sampled_global_embeddings_dict = {}
    for collection_name in seed_dict.keys():
        print(collection_name)
        with open(join('search', collection_name.lower(), 'sampled_embeddings_dict', f'sampled_embeddings_dict_{seed_dict[collection_name]}.pkl'), 'rb') as f:
            sampled_embeddings_dict = pickle.load(f)
            
        np.random.seed(seed_global)
        sampled_indices = np.random.choice(len(sampled_embeddings_dict['_id']), sampling_size, replace=False)    
        
        if len(sampled_global_embeddings_dict.keys())==0:
            sampled_global_embeddings_dict['_id'] = [sampled_embeddings_dict['_id'][t] for t in sampled_indices]
            sampled_global_embeddings_dict['embeddings'] = sampled_embeddings_dict['embeddings'][sampled_indices]
            sampled_global_embeddings_dict['raw_message'] = [sampled_embeddings_dict['raw_message'][t] for t in sampled_indices]
        else:
            sampled_global_embeddings_dict['_id'] += [sampled_embeddings_dict['_id'][t] for t in sampled_indices]
            sampled_global_embeddings_dict['embeddings'] = torch.vstack((sampled_global_embeddings_dict['embeddings'], sampled_embeddings_dict['embeddings'][sampled_indices]))
            sampled_global_embeddings_dict['raw_message'] += [sampled_embeddings_dict['raw_message'][t] for t in sampled_indices]
            
        del sampled_embeddings_dict
        gc.collect()
        
    with open(f'{join("search", "global", "sampled_embeddings_dict", f"sampled_embeddings_dict_{seed_global}")}.pkl', 'wb') as f:
        pickle.dump(sampled_global_embeddings_dict, f)

### B. Title sampling dict

In [15]:

# load articles_by_month from pickle file
articles_by_day = {}
collection_name = 'Thehill'
threshold = 10
model_name = MODEL_NAMES[collection_name]
file_names = os.listdir(join('article', collection_name.lower(), model_name, 'articles_by_day'))
file_names = sorted(file_names, key=lambda x: datetime.strptime(x.split('.')[0], '%Y-%m-%d'))


In [17]:
sampled_title_embeddings_dict = {}
title_embeddings_list = []

for file_name in file_names:
    day = file_name.split('.')[0]
    print(day)
    articles = pd.read_parquet(join('article', collection_name.lower(), model_name, 'articles_by_day', file_name))
    articles = articles[articles['comment_id'].apply(len) > threshold]
    if len(articles)>0:
        title_embeddings_list.append(articles[['_id', 'clean_title', 'title_embeddings']])

2012-07-19
2012-07-20
2012-07-21
2012-07-22
2012-07-23
2012-07-24
2012-07-25
2012-07-26
2012-07-27
2012-07-28
2012-07-29
2012-07-30
2012-07-31
2012-08-01
2012-08-02
2012-08-03
2012-08-04
2012-08-05
2012-08-06
2012-08-07
2012-08-08
2012-08-09
2012-08-10
2012-08-11
2012-08-12
2012-08-13
2012-08-14
2012-08-15
2012-08-16
2012-08-17
2012-08-18
2012-08-19
2012-08-20
2012-08-21
2012-08-22
2012-08-23
2012-08-24
2012-08-25
2012-08-26
2012-08-27
2012-08-28
2012-08-29
2012-08-30
2012-08-31
2012-09-01
2012-09-02
2012-09-03
2012-09-04
2012-09-05
2012-09-06
2012-09-07
2012-09-08
2012-09-09
2012-09-10
2012-09-11
2012-09-12
2012-09-13
2012-09-14
2012-09-15
2012-09-16
2012-09-17
2012-09-18
2012-09-19
2012-09-20
2012-09-21
2012-09-22
2012-09-23
2012-09-24
2012-09-25
2012-09-26
2012-09-27
2012-09-28
2012-09-29
2012-09-30
2012-10-01
2012-10-02
2012-10-03
2012-10-04
2012-10-05
2012-10-06
2012-10-07
2012-10-08
2012-10-09
2012-10-10
2012-10-11
2012-10-12
2012-10-13
2012-10-14
2012-10-15
2012-10-16
2012-10-17

In [18]:
title_embeddings_df = pd.concat(title_embeddings_list, ignore_index=True)
title_embeddings_df.to_feather(join('article', collection_name.lower(), model_name, 'title_embeddings_df.feather'))
print(len(title_embeddings_df))

# Atlantic : 32032
# Breitbart : 
# Gatewaypundit : 80381
# Motherjones : 30967
# Thehill : 307959

307959


In [ ]:
collection_names = list(DATE_RANGES.keys())
seed_list = [1, 2, 3, 4, 5]
sample_size = 30967 # Motherjones (minimum number of articles)

for seed in seed_list:
    title_df_list = []
    
    for collection_name in collection_names:
        print(collection_name)
        model_name = MODEL_NAMES[collection_name]
        title_embeddings_df = pd.read_feather(join('article', collection_name.lower(), model_name, 'title_embeddings_df.feather'))
        # sampling
        np.random.seed(seed)
        sampled_indices = np.random.choice(len(title_embeddings_df['_id']), sample_size, replace=False)
        title_embeddings_df = title_embeddings_df.iloc[sampled_indices]
        title_df_list.append(title_embeddings_df)
    
    title_df = pd.concat(title_df_list, ignore_index=True)
    with open(join('search', 'title', 'sampled_embeddings_dict', f'sampled_embeddings_dict_{seed}.pkl'), 'wb') as f:
        pickle.dump(title_df, f)


## 6. Global ↔ Community model comparison

In [ ]:
topic_embeddings_list = []

for collection_name in COLLECTION_NAMES:
    model_name = MODEL_NAMES[collection_name]
    with open(join("result", collection_name.lower(), model_name+'_output.pkl'), 'rb') as f:
        output = pickle.load(f)
        topic_embeddings_list.append(output['topic_embeddings'])
    
# add global
with open(join("result", "global", MODEL_NAMES["global"]+"_output.pkl"), 'rb') as f:
    output = pickle.load(f)
    global_topic_embedding = output['topic_embeddings']

In [ ]:
cos_sim_list_embedding = []
matching_list_embedding = []

for i in range(len(topic_embeddings_list)):
    dense_list = [topic_embeddings_list[i], global_topic_embedding]
    matching, score_list = cos_sim_between_models(dense_list, None, None, 'embeddings', 0, 1)
    cos_sim_list_embedding.append(sorted(score_list, reverse=True))
    matching_list_embedding.append(matching)

In [ ]:
# plot cos_sim_list_embedding each line with different color
for i in range(len(cos_sim_list_embedding)):
    plt.plot(cos_sim_list_embedding[i], label=COLLECTION_NAMES[i], alpha=0.5)
plt.legend()

In [ ]:
# save cos_sim_list_embedding, matching_list_embedding 
with open(join('result', 'global', 'global_matching_dict.pkl'), 'wb') as f:
    pickle.dump({'cos_sim_list_embedding': cos_sim_list_embedding, 'matching_list_embedding': matching_list_embedding}, f)